In [55]:
homedir = '/home/annzhou/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

test leafspine + unv1

In [56]:
stime = 80 # ms
nlinks = 2048 # 64*16*2, uni-directional
nhosts = 3072
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
seed_list = range(1,6)

In [37]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime / 1000}, ratio {(bw * stime / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 107374080.0, ratio 0.000662652185048191


In [57]:
# generate connection_matrices file (2)
random.seed(0)
ratio = 0.0007
for load in load_list:
    totalbytes = bw * stime / 1000 * load / 100  # B
    mult = totalbytes / unv1bytes / ratio
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_stime{stime}_nlinks{nlinks}_load{load}_nhosts{nhosts}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            for line in lines:
                if random.random() < ratio:
                    tokens = line.split(',')
                    interval = int(tokens[0])
                    fromserver = int(tokens[1])
                    toserver = int(tokens[2])
                    multbytes = int(tokens[3]) * mult

                    # generate flows
                    mybytes_sum = 0
                    while mybytes_sum < multbytes:
                        mybytes = genflowbytes()
                        while mybytes<0 or mybytes>large_flow_threshold:
                            mybytes = genflowbytes()
                        mybytes = adjustbytesbymtu(mybytes)
                        if mybytes_sum + mybytes > multbytes:
                            mybytes = multbytes - mybytes_sum
                            mybytes = adjustbytesbymtu(mybytes)
                            break
                        mybytes_sum += mybytes

                        # generate random start time
                        start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                        fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                        actualbytes += int(mybytes)

                    mybytes_sum += mybytes

                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(mybytes)

                    # print(f'multbytes {multbytes}, mybytes_sum {mybytes_sum}')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 20%, totalbytes 21474816.0, unv1bytes 162036861000, mult 0.18932919572805457, actualbytes 27807000
load 40%, totalbytes 42949632.0, unv1bytes 162036861000, mult 0.37865839145610913, actualbytes 46596000
load 60%, totalbytes 64424448.0, unv1bytes 162036861000, mult 0.5679875871841636, actualbytes 70780500


In [ ]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/test_general_setup/unv1_leafspine.conf'
for seed in seed_list:
    for load in load_list:
        cmfile = f'experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_stime{stime}_nlinks{nlinks}_load{load}_nhosts{nhosts}.cm'
        f.write(f"./run.sh LEAFSPINE {make_leafspine} 80 3072 64 leafspine_{trafficname} EVAL null {multstr} 4 8 0 0 0 0 ecmp 0 {trafficname} {sseed} netpathfiles/netpath_ecmp_leafspine.txt qvarfiles/qvar_leafspine_0_0_ecmp_64 64 50 150 200 0 0 > m_leafspine_{trafficname}_{multname}_{sseed}.log\n")

In [ ]:
trafficname = "unv1"
multstrarr1 = ["0 2 25","0 17 100","0 1 4","0 17 50"]
multstrarr2 = ["0 21 50","0 51 100","0 59 100","0 34 50","0 38 50","0 17 20"]
multnamearr1 = range(5,21,5)
multnamearr2 = range(25,51,5)

conffile = f"/home/annzhou/DRing/src/emp/datacentre/experiments/routing/{trafficname}.conf"
serverfile = f"serverfiles/dring_2988_80_64"
make_leafspine = "MAKE"
make_dring = "MAKE"
with open(conffile,'w') as f:
    for imultstr,multstr in enumerate(multstrarr1):
        for sseed in sseedarr:
            multname = multnamearr1[imultstr]
            # f.write(f"./run.sh LEAFSPINE {make_leafspine} 80 3072 64 leafspine_{trafficname} NEW_WISC null {multstr} 4 8 0 0 0 0 ecmp 0 {trafficname} {sseed} netpathfiles/netpath_ecmp_leafspine.txt qvarfiles/qvar_leafspine_0_0_ecmp_64 64 50 150 200 0 0 > m_leafspine_{trafficname}_{multname}_{sseed}.log\n")
            make_leafspine = "NOMAKE"
            f.write(f"./run.sh RRG {make_dring} 80 2988 64 dring_{trafficname} NEW_WISC graphfiles/ring_supergraph/double_ring/instance1_80_64.edgelist {multstr} 4 8 0 0 0 0 null 0 {trafficname} serverfiles/dring_2988_80_64 {sseed} null null 64 50 150 200 0 0 > m_dring_{trafficname}_{multname}_{sseed}.log\n")
            make_dring = "NOMAKE"
    for imultstr,multstr in enumerate(multstrarr2):
        for sseed in sseedarr:
            multname = multnamearr2[imultstr]
            f.write(f"./run.sh RRG {make_dring} 80 2988 64 dring_{trafficname} NEW_WISC graphfiles/ring_supergraph/double_ring/instance1_80_64.edgelist {multstr} 4 8 0 0 0 0 null 0 {trafficname} serverfiles/dring_2988_80_64 {sseed} null null 64 50 150 200 0 0 > m_dring_{trafficname}_{multname}_{sseed}.log\n")
            make_dring = "NOMAKE"

test leafspine + incast

In [58]:
stime = 80 # ms
nlinks = 2048 # 64*16*2, uni-directional
nhosts = 3072
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
incast_degree = 1000
seed_list = range(1,6)

In [67]:
# generate connection_matrices file
random.seed(0)
dst_host = 0
src_hosts = random.sample(range(1, nhosts), incast_degree)

for load in load_list:
    totalbytes = bw * stime / 1000 * load / 100  # B
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/incast_degree{incast_degree}_stime{stime}_nlinks{nlinks}_load{load}_nhosts{nhosts}.cm'
    with open(cmfile, 'w') as fw:
        done = False
        while actualbytes < totalbytes and not done:
            for src_host in src_hosts:
                # generate flows
                mybytes = genflowbytes()
                while mybytes<0 or mybytes>large_flow_threshold:
                    mybytes = genflowbytes()
                mybytes = adjustbytesbymtu(mybytes)
                if actualbytes + mybytes > totalbytes:
                    mybytes = totalbytes - actualbytes
                    mybytes = adjustbytesbymtu(mybytes)
                    done = True
                    break

                # generate random start time
                start_time_ms = random.uniform(0, stime)

                fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
                actualbytes += int(mybytes)
            
        # generate random start time
        start_time_ms = random.uniform(0, stime)

        fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
        actualbytes += int(mybytes)

    print(f'load {load}%, totalbytes {totalbytes}, actualbytes {actualbytes}')


load 20%, totalbytes 21474816.0, actualbytes 21475500
load 40%, totalbytes 42949632.0, actualbytes 42951000
load 60%, totalbytes 64424448.0, actualbytes 64425000


test dring + unv1

test dring + incast